# Day 11 — Optimizers

## 1. Learning Objectives
- Understand how `torch.optim` replaces manual parameter updates.
- Learn the differences between **SGD**, **Momentum**, **Adam**, and **AdamW**.
- Implement an optimizer in a training loop.
- Understand the Learning Rate (`lr`) hyperparameter intuitively.

## 2. Prerequisites
- Autograd (Day 5)
- Manual Training Loop (Day 6)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

## 3. Concept Explanation
In Day 6, we manually updated weights like this:
`w -= learning_rate * w.grad`

This is vanilla **Gradient Descent**. But this approach has flaws:
1. It is slow to converge.
2. It easily gets stuck in "local minima" (small valleys in the loss landscape) and stops learning.

`torch.optim` provides advanced mathematical algorithms (Optimizers) that use the gradients to update the weights in much smarter ways.

## 5. Intuition: The 4 Main Optimizers
Imagine you are a blindfolded hiker trying to find the bottom of a valley.

1. **SGD (Stochastic Gradient Descent)**: You feel the slope with your foot and take a single step downhill. Very basic.
2. **SGD with Momentum**: You start rolling a boulder down the hill. It builds up speed, allowing it to roll *over* small bumps (local minima) that would have stopped standard SGD.
3. **Adam (Adaptive Moment Estimation)**: You are a smart hiker who adjusts your step size. If a path is consistently steep, you take smaller, careful steps. If a path is flat, you take bigger steps to speed up. Adam is the default industry standard.
4. **AdamW**: Adam, but with a better "Weight Decay" (regularization) implementation. Highly recommended for modern networks.

## 8. Simple Example: Using an Optimizer
Let's see how `torch.optim` cleans up our code.

In [ ]:
# Dummy model and data
model = nn.Linear(10, 1)
dummy_X = torch.randn(5, 10)
dummy_y = torch.randn(5, 1)
criterion = nn.MSELoss()

# 1. Initialize the Optimizer
# We MUST pass the model's parameters to the optimizer so it knows what to update!
optimizer = optim.Adam(model.parameters(), lr=0.01)

# --- Inside the Training Loop ---

# Forward Pass
predictions = model(dummy_X)
loss = criterion(predictions, dummy_y)

# Backward Pass
loss.backward()

# THE MAGIC HAPPENS HERE:
# 1. Optimizer steps (updates all weights)
optimizer.step()

# 2. Zero the gradients (replaces w.grad.zero_())
optimizer.zero_grad()

print("Weights updated successfully!")

## 9. Code Walkthrough
- `optim.Adam(model.parameters(), lr=0.01)`: We hand over control of our model's weights to the Adam algorithm. The Learning Rate (`lr`) determines the base step size.
- `optimizer.step()`: The optimizer looks at `.grad` for every parameter and applies the Adam math formula to update them.
- `optimizer.zero_grad()`: Clears out all the gradients across the entire model instantly.

## 11. Practice Exercise 1: Instantiating Optimizers
Create a simple Sequential model with two Linear layers. 
Instantiate an `optim.SGD` optimizer for it with a learning rate of `0.05` and a momentum of `0.9`.

In [ ]:
# Write your code here

In [ ]:
# SOLUTION
prac_model = nn.Sequential(
    nn.Linear(20, 10),
    nn.ReLU(),
    nn.Linear(10, 2)
)

prac_optim = optim.SGD(prac_model.parameters(), lr=0.05, momentum=0.9)
print(prac_optim)

## 13. Debugging Challenge
Why will the weights of `model_b` NEVER update in this code, even though we call `optimizer.step()`?

In [ ]:
model_a = nn.Linear(5, 1)
model_b = nn.Linear(5, 1)

opt = optim.Adam(model_a.parameters(), lr=0.01)

dummy_in = torch.randn(2, 5)
loss = model_b(dummy_in).sum() # Note: we are passing data through model_b
loss.backward()

opt.step()
opt.zero_grad()

**Solution:** The optimizer `opt` was initialized with `model_a.parameters()`. However, the data was passed through `model_b`, meaning `model_b`'s gradients were computed during `.backward()`. The optimizer only updates the parameters it was explicitly given (`model_a`), which have no gradients! You must pass the correct model parameters to the optimizer.

## 17. Interview Questions
1. **Why is Adam generally preferred over standard SGD?**
   *Answer*: Adam computes individual adaptive learning rates for different parameters. It converges much faster and is generally less sensitive to the exact choice of the initial learning rate compared to vanilla SGD.
2. **What does `optimizer.zero_grad()` actually do?**
   *Answer*: It iterates through all the parameters passed to the optimizer during initialization and sets their `.grad` attribute to zero (or None), preventing gradients from accumulating across training iterations.

## 19. Day Summary
- Never update weights manually using `with torch.no_grad():`. Use `torch.optim`.
- **Adam/AdamW** is your go-to optimizer for 95% of Deep Learning tasks.
- Remember to pass `model.parameters()` into the optimizer.
- The new loop end: `optimizer.step()` followed by `optimizer.zero_grad()`.